In [1]:
import pandas as pd
import numpy as np

# 1. Loading Data and initialization of Silver Layer

In [2]:
file_path = r'C:\Users\Maciek\Desktop\netflixdb\databases\users.csv'

df_bronze = pd.read_csv(file_path, sep = ',')

df_silver = df_bronze.copy()

df_silver.head()

,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,subscription_start_date,is_active,monthly_spend,primary_device,household_size,created_at
0,user_00001,figueroajohn@example.org,Erica,Garza,43.0,Male,USA,Massachusetts,North Jefferyhaven,Basic,2024-04-08,True,36.06,Laptop,1.0,2023-04-01 14:40:50.540242
1,user_00002,blakeerik@example.com,Joshua,Bernard,38.0,Male,USA,Texas,North Noahstad,Premium+,2024-05-24,True,14.59,Desktop,2.0,2024-10-10 15:39:11.030515
2,user_00003,smiller@example.net,Barbara,Williams,32.0,Female,USA,Michigan,Traciebury,Standard,2023-09-22,False,11.71,Desktop,3.0,2024-06-29 14:27:49.560875
3,user_00004,mitchellclark@example.com,Chelsea,Ferguson,11.0,Male,USA,Ohio,South Noah,Standard,2024-08-21,True,28.56,Laptop,2.0,2023-04-11 01:01:59.614841
4,user_00005,richard13@example.net,Jason,Foster,21.0,Female,USA,Arizona,West Donald,Standard,2024-10-28,True,9.54,Desktop,6.0,2025-04-12 19:59:30.137806


# 2.Analyses and Exploration Nulls

In [3]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  10300 non-null  object 
 1   email                    10300 non-null  object 
 2   first_name               10300 non-null  object 
 3   last_name                10300 non-null  object 
 4   age                      9071 non-null   float64
 5   gender                   9476 non-null   object 
 6   country                  10300 non-null  object 
 7   state_province           10300 non-null  object 
 8   city                     10300 non-null  object 
 9   subscription_plan        10300 non-null  object 
 10  subscription_start_date  10300 non-null  object 
 11  is_active                10300 non-null  bool   
 12  monthly_spend            9283 non-null   float64
 13  primary_device           10300 non-null  object 
 14  household_size        

In [4]:
df_silver.isnull().sum()

user_id                       0
email                         0
first_name                    0
last_name                     0
age                        1229
gender                      824
country                       0
state_province                0
city                          0
subscription_plan             0
subscription_start_date       0
is_active                     0
monthly_spend              1017
primary_device                0
household_size             1545
created_at                    0
dtype: int64

In [ ]:
# Analyze email domains
df_silver['email_domain'] = df_silver['email'].str.split('@').str[1]

domain_counts = df_silver['email_domain'].value_counts()
print(domain_counts)

email_domain
example.org    3490
example.com    3449
example.net    3361
Name: count, dtype: int64


# 3. Droping duplicates

In [ ]:
# Check for duplicate user_id entries
duplicate_count = df_silver['user_id'].duplicated().sum()
print(f"Liczba duplikatów movie_id: {duplicate_count}")


Liczba duplikatów movie_id: 300


In [ ]:
# Display duplicate user_id entries
df_duplicates = df_silver[df_silver['user_id'].duplicated(keep=False)]
print(df_duplicates.sort_values('user_id'))

          user_id                   email first_name last_name   age  gender  \
27     user_00028       rsims@example.com   Jennifer    Nelson   NaN    Male   
10023  user_00028       rsims@example.com   Jennifer    Nelson   NaN    Male   
10216  user_00115    jelliott@example.com   Jennifer     Scott  30.0    Male   
114    user_00115    jelliott@example.com   Jennifer     Scott  30.0    Male   
10079  user_00120     dpeters@example.com     Kristi    Horton  39.0    Male   
...           ...                     ...        ...       ...   ...     ...   
9956   user_09957  qfernandez@example.org     Daniel   Morales  26.0    Male   
9957   user_09958   debraking@example.com      Amber  Phillips  35.0    Male   
10299  user_09958   debraking@example.com      Amber  Phillips  35.0    Male   
10180  user_09981     klarsen@example.net    Bradley      Leon  47.0  Female   
9980   user_09981     klarsen@example.net    Bradley      Leon  47.0  Female   

      country        state_province    

In [ ]:
# Remove duplicate user_id entries, keeping the first occurrence
df_silver = df_silver.drop_duplicates(keep = 'first')

# 4. Cleaning and Imputation of data in 'age' column

In [ ]:
print(df_silver['age'].value_counts())

# Handle invalid age values
median_value = df_silver['age'].median()
print(f'Median age: {median_value}')

# Set invalid ages to NaN
df_silver.loc[(df_silver['age'] < 0 ) | (df_silver['age']>= 100), 'age'] = np.nan

# Create indicator for missing age values
df_silver['is_age_missing'] = df_silver['age'].isnull().astype(int)

# Impute missing age values with median
df_silver['age'] = df_silver['age'].fillna(median_value)

age
 35.0     308
 31.0     296
 36.0     286
 33.0     282
 38.0     277
         ... 
 107.0      1
 75.0       1
-7.0        1
 95.0       1
 93.0       1
Name: count, Length: 97, dtype: int64
Median age: 35.0


In [11]:
df_silver['gender'].value_counts()

gender
Female               4203
Male                 4096
Prefer not to say     456
Other                 445
Name: count, dtype: int64

# 5. Cleaning and Imputation of caterogical data in 'gender' column

In [ ]:
new_category = 'UNSPECIFIED'

recode_map = {
    'Prefer not to say': new_category,
    'Other': new_category
}
# Recode gender values  
df_silver['gender'] = df_silver['gender'].replace(recode_map)

# Create indicator
df_silver['is_gender_missing'] = df_silver['gender'].isnull().astype(int)

# Impute missing gender values
df_silver['gender'] = df_silver['gender'].fillna(new_category)

#Verification
df_silver['gender'].value_counts()

gender
Female         4203
Male           4096
UNSPECIFIED    1701
Name: count, dtype: int64

# 6. Cleaning and Standarization of empirical data in 'monthly_spend' column

In [ ]:

median_value1 = df_silver['monthly_spend'].median()
print(f'Median monthly spend: {median_value1}')

Median monthly spend: 13.56


In [ ]:
# Create indicator for missing monthly_spend values
df_silver['is_monthly_spend_missing'] = df_silver['monthly_spend'].isnull().astype(int)

# Impute missing monthly_spend values with median
df_silver['monthly_spend'] = df_silver['monthly_spend'].fillna(median_value1)

In [15]:
df_silver['household_size'].value_counts()

household_size
2.0    2582
1.0    1667
3.0    1663
4.0    1311
5.0     677
6.0     364
7.0     166
8.0      70
Name: count, dtype: int64

# 7. Cleaning and Standarization of caterogical data in 'household_size' column

In [16]:
median_value2 = df_silver['household_size'].median()
print(f'Median household size: {median_value2}')

Median household size: 3.0


In [ ]:

# Create indicator for missing household_size values
df_silver['is_household_size_missing'] = df_silver['household_size'].isnull().astype(int)
df_silver['household_size'] = df_silver['household_size'].fillna(median_value2)

# 8. Formating data types

In [18]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   user_id                    10000 non-null  object 
 1   email                      10000 non-null  object 
 2   first_name                 10000 non-null  object 
 3   last_name                  10000 non-null  object 
 4   age                        10000 non-null  float64
 5   gender                     10000 non-null  object 
 6   country                    10000 non-null  object 
 7   state_province             10000 non-null  object 
 8   city                       10000 non-null  object 
 9   subscription_plan          10000 non-null  object 
 10  subscription_start_date    10000 non-null  object 
 11  is_active                  10000 non-null  bool   
 12  monthly_spend              10000 non-null  float64
 13  primary_device             10000 non-null  object 
 

In [23]:
#Formating age column to integer
df_silver['age'] = df_silver['age'].astype(int)

#Formating subscription_start_date to datetime
df_silver['subscription_start_date'] = pd.to_datetime(df_silver['subscription_start_date'], errors='coerce')
df_silver['subscription_start_date'] = df_silver['subscription_start_date'].dt.strftime('%Y-%m-%d')

#Formating household_size to integer
df_silver['household_size'] = df_silver['household_size'].astype(int)

In [24]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   user_id                    10000 non-null  object 
 1   email                      10000 non-null  object 
 2   first_name                 10000 non-null  object 
 3   last_name                  10000 non-null  object 
 4   age                        10000 non-null  int64  
 5   gender                     10000 non-null  object 
 6   country                    10000 non-null  object 
 7   state_province             10000 non-null  object 
 8   city                       10000 non-null  object 
 9   subscription_plan          10000 non-null  object 
 10  subscription_start_date    10000 non-null  object 
 11  is_active                  10000 non-null  bool   
 12  monthly_spend              10000 non-null  float64
 13  primary_device             10000 non-null  object 
 

# 9. Saving Silver Layer to CSV

In [25]:
output_path = 'netflix_silver_layer_users.csv'

df_silver.to_csv(
    output_path,
    index = False,
    sep = ','
)